In [ ]:
!pip install qiskit
!pip install qiskit_aer

In [4]:
import qiskit
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import AerSimulator

# 1. Definición de Registros (3 cúbits de datos, 2 cúbits ancila para estabilizadores)
data_reg = QuantumRegister(3, name='data')
ancilla_reg = QuantumRegister(2, name='ancilla')
syndrome_reg = ClassicalRegister(2, name='syndrome')
measure_reg = ClassicalRegister(3, name='measure')

circuit = QuantumCircuit(data_reg, ancilla_reg, syndrome_reg, measure_reg)

# 2. Estado Inicial: Preparar el cúbit de datos 0 en el estado |1> como prueba
circuit.x(data_reg[0])
circuit.barrier()

# 3. Codificación: Crear el estado entrelazado |111> (Repetition Code)
circuit.cx(data_reg[0], data_reg[1])
circuit.cx(data_reg[0], data_reg[2])
circuit.barrier()

# 4. Inyección del Error: Simulamos un error X en el cúbit de datos 1
# Cambia el índice para probar el comportamiento con otros cúbits
circuit.x(data_reg[1])
circuit.barrier()

# 5. Generadores de Estabilizadores: Medición de Z_0 Z_1 y Z_1 Z_2
# Estabilizador 1: Z_0 Z_1 (Detecta diferencias entre cúbit 0 y 1)
circuit.cx(data_reg[0], ancilla_reg[0])
circuit.cx(data_reg[1], ancilla_reg[0])

# Estabilizador 2: Z_1 Z_2 (Detecta diferencias entre cúbit 1 y 2)
circuit.cx(data_reg[1], ancilla_reg[1])
circuit.cx(data_reg[2], ancilla_reg[1])

# Medición del síndrome de error
circuit.measure(ancilla_reg, syndrome_reg)
circuit.barrier()

# 6. Corrección de Errores (Lógica del Síndrome basada en la medición)
# Síndrome '01' (en binario Qiskit: ancilla 1 es 0, ancilla 0 es 1) -> Error en cúbit 0
with circuit.if_test((syndrome_reg, 1)):
    circuit.x(data_reg[0])

# Síndrome '11' (ancilla 1 es 1, ancilla 0 es 1) -> Error en cúbit 1
with circuit.if_test((syndrome_reg, 3)):
    circuit.x(data_reg[1])

# Síndrome '10' (ancilla 1 es 1, ancilla 0 es 0) -> Error en cúbit 2
with circuit.if_test((syndrome_reg, 2)):
    circuit.x(data_reg[2])

circuit.barrier()

# 7. Medición Final de los cúbits de datos para verificar la corrección
circuit.measure(data_reg, measure_reg)

# 8. Ejecución en el Simulador
simulator = AerSimulator()
compiled_circuit = qiskit.transpile(circuit, simulator)
result = simulator.run(compiled_circuit, shots=1000).result()
counts = result.get_counts()

print("Resultados de la medición (Formato: 'medición_datos síndrome'):")
print(counts)


Resultados de la medición (Formato: 'medición_datos síndrome'):
{'111 11': 1000}


La medición de los datos devuelve '111' y el síndrome devuelve '11'. Al ser el síndrome '11', el bloque condicional with circuit.if_test((syndrome_reg, 3)): detecta el valor entero 3 y ejecuta una compuerta de corrección X sobre el cúbits data[1] que es donde etaba el error.